**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/Users/kavyajeetbora@gmail.com/fmcg_databricks/consolidated_pipeline/dimension_data_processing/00_config

In [0]:
# dbutils.widgets.text("catalog", "fmcg", "Catalog")
# dbutils.widgets.text("data_source", "orders", "Data Source")
# dbutils.widgets.text("s3", "sportsbar-raw-data-5684", "S3 Bucket")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
s3_bucket = dbutils.widgets.get('s3')

base_path = f"s3://{s3_bucket}/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

## Bronze


Now imagine today is 2nd December and we have received the 1st December data dump in the s3 bucket as csv
we will ingest it as staging table in bronze

In [0]:
bronze_staging_df = (
    spark.read.options(header=True, inferSchema=True)
    .csv(f"{landing_path}/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total Rows: ", bronze_staging_df.count())
bronze_staging_df.show(5)

### Staging table to process just the arrived incremenal data

**Note:**  Here we are going to `append` not `upsert` because in bronze, it meant to write files as it is, no processing (that only belongs to silver stage)

Bronze protects against data loss, Silver protects against bad data. Upserting at Bronze quietly moves Silver's job upstream and gives up the "replay everything from scratch" guarantee that makes the medallion architecture useful in the first place

In [0]:
(
    bronze_staging_df.write.format('delta')
    .option("delta.enableChangeDataFeed", True)
    .mode('append')
    .saveAsTable(bronze_table)
)

In [0]:
(
    bronze_staging_df.write.format('delta')
    .option("delta.enableChangeDataFeed", True)
    .mode('overwrite')
    .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")
)


### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(file_info.path, f"{processed_path}/{file_info.name}", True)

## Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};")
df_orders.show(2)

**Transformations**

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
    .otherwise("999999")
    .cast("string"),
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", ""),
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    ),
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(
    ["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"]
)

# 5. convert product id to string
df_orders = df_orders.withColumn("product_id", F.col("product_id").cast("string"))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date"),
).show()

**Join with products**

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(
    df_orders["*"], df_products["product_code"]
)

df_joined.show(5)

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true").option(
        "mergeSchema", "true"
    ).mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(
        df_joined.alias("bronze"),
        "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Staging table to process just the arrived incremenal data

In [0]:
# stagging for incremental data
df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true").mode(
    "overwrite"
).saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

## Gold

In [0]:
df_gold = spark.sql(
    f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {catalog}.{silver_schema}.staging_{data_source};"
)

df_gold.show(2)

In [0]:
df_gold.count()

Upserting the Child Table Updates

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option("delta.enableChangeDataFeed", "true").option(
        "mergeSchema", "true"
    ).mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(
        df_gold.alias("gold"),
        "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Merging with Parent company

- Since the data in parent table is in monthly level aggregate and our child table is in daily level
- We will filter the data from child that belongs to the incremental load and then aggregate 
- And Finally upsert the values to the parent table

In [0]:
# First Identify the incremental month

incremental_month_df = (
    spark.sql(
        f"SELECT order_placement_date as date FROM {catalog}.{silver_schema}.staging_{data_source}"
    )
    .select(F.trunc(F.col("date").cast("date"), "MM").alias("start_month_unique"))
    .distinct()
)
display(incremental_month_df)

In [0]:
## Now filter the child table orders data for that incremental month
sb_fact_orders_df = (
    spark.sql(f"SELECT date, product_code, customer_code,sold_quantity FROM {catalog}.{gold_schema}.sb_fact_orders")
    .withColumn("start_month", F.trunc(F.col("date").cast("date"), "MM"))
)

sb_fact_orders_incremental_df = (
    sb_fact_orders_df.alias("T1")
    .join(
        incremental_month_df.alias("T2"),
        on = F.col("T1.start_month") == F.col("T2.start_month_unique"),
        how = "inner"
    )
)

print("Date Range to be Updated in Child Table:")
sb_fact_orders_incremental_df.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).show()

Aggregate the data at monthly level

In [0]:
sb_fact_orders_monthly_agg_df = (
    sb_fact_orders_incremental_df
    .groupby("start_month", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("start_month", "date")
)

display(sb_fact_orders_monthly_agg_df.limit(10))

In [0]:
sb_fact_orders_monthly_agg_df.count()

Upsert the Incremental Child Gold Data to Parent's Gold Data

In [0]:
## Read the Parent table where incremental data is to me upserted
gold_parent_data = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")

## Now merge

(
    gold_parent_data.alias("parent_gold")
    .merge(
        sb_fact_orders_monthly_agg_df.alias("child_staging_gold"),
        "parent_gold.date = child_staging_gold.date AND parent_gold.product_code = child_staging_gold.product_code AND parent_gold.customer_code = child_staging_gold.customer_code"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

## Cleanup

In [0]:
%sql
DROP TABLE IF EXISTS fmcg.bronze.staging_orders;

In [0]:
%sql
DROP TABLE IF EXISTS fmcg.silver.staging_orders;